# 03 Evaluation
This notebook evaluates the best checkpoint on the exact saved test split, writes detailed reports, and generates qualitative visual diagnostics.


In [ ]:
import csv
import json
import random
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

# Configuration
DATA_DIR = Path("data")
MODELS_DIR = Path("models")
RESULTS_DIR = Path("results")
CHECKPOINT_PATH = MODELS_DIR / "densenetAllBest.pth"
SPLIT_FILE = MODELS_DIR / "split_indices.json"
CLASS_NAMES_FILE = MODELS_DIR / "class_names.txt"

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
SEED = 42
ENABLE_GRADCAM = True
GRADCAM_MAX_CORRECT = 4
GRADCAM_MAX_WRONG = 4

(RESULTS_DIR / "confusion_matrix").mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "gradcam").mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "sample_predictions").mkdir(parents=True, exist_ok=True)


In [ ]:
# Device + reproducibility
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print(f"Using device: {device}")


In [ ]:
class EvalSubset(Dataset):
    def __init__(self, base_dataset, indices, transform):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        sample_idx = self.indices[idx]
        path, target = self.base_dataset.samples[sample_idx]
        image = self.base_dataset.loader(path)
        if self.transform is not None:
            image = self.transform(image)
        return image, target, path

def stratified_split_indices(targets, train_ratio, val_ratio, seed):
    per_class_indices = defaultdict(list)
    for idx, label in enumerate(targets):
        per_class_indices[label].append(idx)

    rng = random.Random(seed)
    train_indices, val_indices, test_indices = [], [], []
    for _, cls_indices in per_class_indices.items():
        rng.shuffle(cls_indices)
        n_total = len(cls_indices)
        n_train = int(n_total * train_ratio)
        n_val = int(n_total * val_ratio)

        train_indices.extend(cls_indices[:n_train])
        val_indices.extend(cls_indices[n_train:n_train + n_val])
        test_indices.extend(cls_indices[n_train + n_val:])

    rng.shuffle(train_indices)
    rng.shuffle(val_indices)
    rng.shuffle(test_indices)
    return train_indices, val_indices, test_indices

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data directory not found: {DATA_DIR.resolve()}")

base_dataset = datasets.ImageFolder(root=str(DATA_DIR))
targets = [label for _, label in base_dataset.samples]

if CLASS_NAMES_FILE.exists():
    class_names = [line.strip() for line in CLASS_NAMES_FILE.read_text(encoding="utf-8").splitlines() if line.strip()]
else:
    class_names = base_dataset.classes

if SPLIT_FILE.exists():
    split_data = json.loads(SPLIT_FILE.read_text(encoding="utf-8"))
    test_indices = split_data["test_indices"]
    print(f"Loaded split file: {SPLIT_FILE}")
else:
    _, _, test_indices = stratified_split_indices(targets, 0.70, 0.15, SEED)
    print("Split file missing, regenerated test split with seed 42")

test_dataset = EvalSubset(base_dataset, test_indices, eval_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Classes ({len(class_names)}): {class_names}")
print(f"Test set size: {len(test_dataset)}")


In [ ]:
# Model + checkpoint
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH.resolve()}")

try:
    model = models.densenet121(weights=None)
except TypeError:
    model = models.densenet121(pretrained=False)

in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, len(class_names))
model = model.to(device)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Loaded checkpoint: {CHECKPOINT_PATH}")
print(f"Checkpoint epoch: {checkpoint.get('epoch', 'unknown')}")
print(f"Checkpoint val_loss: {checkpoint.get('val_loss', 'unknown')}")


In [ ]:
# Inference
all_labels = []
all_preds = []
all_probs = []
all_paths = []

with torch.no_grad():
    for images, labels, paths in tqdm(test_loader, desc="evaluating", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        all_labels.extend(labels.cpu().tolist())
        all_preds.extend(preds.cpu().tolist())
        all_probs.extend(probs.max(dim=1).values.cpu().tolist())
        all_paths.extend(list(paths))

accuracy = accuracy_score(all_labels, all_preds)
balanced_acc = balanced_accuracy_score(all_labels, all_preds)
macro_f1 = f1_score(all_labels, all_preds, average="macro")

print(f"Accuracy: {accuracy:.4f}")
print(f"Balanced accuracy: {balanced_acc:.4f}")
print(f"Macro F1: {macro_f1:.4f}")


In [ ]:
# Reports
report_text = classification_report(all_labels, all_preds, target_names=class_names, digits=4)
report_dict = classification_report(all_labels, all_preds, target_names=class_names, digits=4, output_dict=True)

print(report_text)

(RESULTS_DIR / "classification_report.txt").write_text(report_text, encoding="utf-8")
(RESULTS_DIR / "classification_report.json").write_text(json.dumps(report_dict, indent=2), encoding="utf-8")

with open(RESULTS_DIR / "predictions.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["path", "true_label", "pred_label", "confidence"])
    for path, true_idx, pred_idx, conf in zip(all_paths, all_labels, all_preds, all_probs):
        writer.writerow([path, class_names[true_idx], class_names[pred_idx], f"{conf:.6f}"])

summary_lines = [
    f"accuracy: {accuracy:.6f}",
    f"balanced_accuracy: {balanced_acc:.6f}",
    f"macro_f1: {macro_f1:.6f}",
    f"checkpoint_epoch: {checkpoint.get('epoch', 'unknown')}",
]
(RESULTS_DIR / "evaluation_summary.txt").write_text("\n".join(summary_lines) + "\n", encoding="utf-8")

print(f"Saved: {(RESULTS_DIR / 'classification_report.txt')}")
print(f"Saved: {(RESULTS_DIR / 'predictions.csv')}")
print(f"Saved: {(RESULTS_DIR / 'evaluation_summary.txt')}")


In [ ]:
# Confusion matrix (counts + normalized)
cm = confusion_matrix(all_labels, all_preds)
cm_norm = confusion_matrix(all_labels, all_preds, normalize="true")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(
    ax=axes[0], cmap="Blues", colorbar=False, xticks_rotation=45
)
axes[0].set_title("Confusion Matrix (Counts)")

ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=class_names).plot(
    ax=axes[1], cmap="Blues", colorbar=False, xticks_rotation=45, values_format=".2f"
)
axes[1].set_title("Confusion Matrix (Normalized)")

plt.tight_layout()
cm_path = RESULTS_DIR / "confusion_matrix" / "confusion_matrix_combined.png"
plt.savefig(cm_path, dpi=300)
plt.show()
plt.close(fig)

print(f"Saved confusion matrix image -> {cm_path}")


In [ ]:
# Sample predictions and errors
def unnormalize(tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = tensor.detach().cpu().numpy().transpose(1, 2, 0)
    img = img * std + mean
    return np.clip(img, 0, 1)

def save_prediction_grid(want_correct=True, max_images=8, file_name="sample_predictions.png"):
    collected = []
    with torch.no_grad():
        for images, labels, _ in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            probs = torch.softmax(model(images), dim=1)
            preds = probs.argmax(dim=1)
            confs = probs.max(dim=1).values

            matches = preds.eq(labels)
            if not want_correct:
                matches = ~matches

            for i in range(images.size(0)):
                if matches[i].item():
                    collected.append((images[i].cpu(), labels[i].item(), preds[i].item(), confs[i].item()))
                if len(collected) >= max_images:
                    break
            if len(collected) >= max_images:
                break

    if not collected:
        print(f"No examples found for want_correct={want_correct}")
        return

    n = len(collected)
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.5 * rows))
    if rows == 1:
        axes = np.array([axes])

    for idx in range(rows * cols):
        ax = axes[idx // cols, idx % cols]
        if idx >= n:
            ax.axis("off")
            continue

        image, y_true, y_pred, conf = collected[idx]
        ax.imshow(unnormalize(image))
        title_color = "green" if y_true == y_pred else "red"
        ax.set_title(
            f"T:{class_names[y_true]}\nP:{class_names[y_pred]} ({conf:.2f})",
            color=title_color,
        )
        ax.axis("off")

    plt.tight_layout()
    out_path = RESULTS_DIR / "sample_predictions" / file_name
    plt.savefig(out_path, dpi=300)
    plt.show()
    plt.close(fig)
    print(f"Saved prediction grid -> {out_path}")

save_prediction_grid(want_correct=True, max_images=8, file_name="correct_examples.png")
save_prediction_grid(want_correct=False, max_images=8, file_name="misclassified_examples.png")


In [ ]:
# Optional Grad-CAM
gradcam_available = False
if ENABLE_GRADCAM:
    try:
        from pytorch_grad_cam import GradCAM
        from pytorch_grad_cam.utils.image import show_cam_on_image
        from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
        gradcam_available = True
    except ImportError:
        print("Grad-CAM package not installed. Run: pip install grad-cam")

def run_gradcam_examples(want_correct=True, max_images=4, prefix="correct"):
    cam = GradCAM(model=model, target_layers=[model.features])
    saved = 0

    for images, labels, _ in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        matches = preds.eq(labels)
        if not want_correct:
            matches = ~matches

        for i in range(images.size(0)):
            if not matches[i].item():
                continue

            input_tensor = images[i].unsqueeze(0)
            pred_idx = preds[i].item()
            true_idx = labels[i].item()
            conf = probs[i, pred_idx].item()

            targets = [ClassifierOutputTarget(pred_idx)]
            grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

            rgb_img = unnormalize(images[i])
            visualization = show_cam_on_image(rgb_img.astype(np.float32), grayscale_cam, use_rgb=True)

            fig, axes = plt.subplots(1, 2, figsize=(8, 4))
            axes[0].imshow(rgb_img)
            axes[0].set_title(f"Original\nT:{class_names[true_idx]}")
            axes[0].axis("off")

            axes[1].imshow(visualization)
            axes[1].set_title(f"Grad-CAM\nP:{class_names[pred_idx]} ({conf:.2f})")
            axes[1].axis("off")

            plt.tight_layout()
            out_file = RESULTS_DIR / "gradcam" / f"{prefix}_{saved + 1}.png"
            plt.savefig(out_file, dpi=300)
            plt.show()
            plt.close(fig)

            saved += 1
            if saved >= max_images:
                return

if gradcam_available:
    try:
        run_gradcam_examples(want_correct=True, max_images=GRADCAM_MAX_CORRECT, prefix="correct")
        run_gradcam_examples(want_correct=False, max_images=GRADCAM_MAX_WRONG, prefix="misclassified")
        print("Saved Grad-CAM examples to results/gradcam")
    except Exception as gradcam_error:
        print(f"Grad-CAM generation skipped due to runtime error: {gradcam_error}")
